In [0]:

from pyspark.sql import SparkSession
from pyspark.sql import functions as F


spark = SparkSession.builder.getOrCreate()


data = [
    (1, "Asha",  55000.0, "IN", "Engineering"),
    (2, "Rohit", 72000.0, "IN", "Data"),
    (3, "Meera", 48000.0, "US", "Support"),
    (4, "Karan", 88000.0, "UK", "Data"),
    (5, "Vijay", None,    "IN", "HR"),     # notice a NULL salary
]

cols = ["id", "name", "salary", "country", "dept"]
df = spark.createDataFrame(data, cols)
df.show()


In [0]:
df.createOrReplaceTempView("employees")

In [0]:
# 1) Total salary of all employees
total_salary_df =df.select(
    F.sum(F.coalesce(F.col("Salary"),F.lit(0.0))).alias("total_salary")
)
total_salary_df.show()

In [0]:
%sql
--1) Total salary of all employees
select sum(coalesce(salary,0,0)) As  total_salary from employees

In [0]:
# 2) Total salary per department
# PySpark

dept_total_df = (df
    .groupBy("dept")
    .agg(F.sum(F.coalesce(F.col("salary"), F.lit(0.0))).alias("dept_total_salary"))
)
dept_total_df.show()

In [0]:
%sql
-- Spark SQL
SELECT
  dept,
  SUM(COALESCE(salary, 0.0)) AS dept_total_salary
FROM employees
GROUP BY dept;

In [0]:
#3) All columns + an extra column with each department’s total salary
# PySpark (Window)
from pyspark.sql.window import Window

w_dept = Window.partitionBy("dept")

df_with_total = (df
    .withColumn(
        "dept_total_salary",
        F.sum(F.coalesce(F.col("salary"), F.lit(0.0))).over(w_dept)
    )
)
df_with_total.show(truncate=False)

In [0]:
#Spark SQL (Window)

SELECT
  id, name, salary, country, dept,
  SUM(COALESCE(salary, 0.0)) OVER (PARTITION BY dept) AS dept_total_salary
FROM employees;

In [0]:
# 4) Percentage of the department salary contributed by each employee
# PySpark

from pyspark.sql.window import Window

w_dept = Window.partitionBy("dept")

df_pct = (df
    .withColumn("salary_0", F.coalesce(F.col("salary"), F.lit(0.0)))
    .withColumn("dept_total_salary", F.sum(F.col("salary_0")).over(w_dept))
    .withColumn(
        "pct_of_dept_salary",
        F.when(F.col("dept_total_salary") == 0, F.lit(0.0))
         .otherwise((F.col("salary_0") / F.col("dept_total_salary")) * 100.0)
    )
    .drop("salary_0")
)
df_pct.show(truncate=False)

In [0]:
%sql
WITH base AS (
  SELECT id, name, dept, COALESCE(salary, 0.0) AS salary_0, country
  FROM employees
),
with_total AS (
  SELECT
    b.*,
    SUM(salary_0) OVER (PARTITION BY dept) AS dept_total_salary
  FROM base b
)
SELECT
  id, name, country, dept,
  salary_0 AS salary,
  dept_total_salary,
  CASE WHEN dept_total_salary = 0 THEN 0.0
       ELSE (salary_0 / dept_total_salary) * 100.0
  END AS pct_of_dept_salary
FROM with_total;